In [1]:
import pandas as pd
import numpy as np

In [2]:
dat = pd.read_csv("E:/Project/customer_churn_dataset-testing-master.csv")
data = dat.copy()

In [3]:
data.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0
1,2,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0
2,3,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0
3,4,35.0,Male,9.0,12.0,5.0,17.0,NaN,Quarterly,232.0,18.0,0.0
4,5,53.0,Female,58.0,24.0,9.0,2.0,NaN,Annual,533.0,18.0,0.0


In [4]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64374 entries, 0 to 64373
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         64374 non-null  int64  
 1   Age                64214 non-null  float64
 2   Gender             64123 non-null  object 
 3   Tenure             64308 non-null  float64
 4   Usage Frequency    64055 non-null  float64
 5   Support Calls      63927 non-null  float64
 6   Payment Delay      63906 non-null  float64
 7   Subscription Type  64096 non-null  object 
 8   Contract Length    64230 non-null  object 
 9   Total Spend        64325 non-null  float64
 10  Last Interaction   64027 non-null  float64
 11  Churn              64027 non-null  float64
dtypes: float64(8), int64(1), object(3)
memory usage: 5.9+ MB
None


In [5]:
print(round(data.describe(), 2))

       CustomerID       Age    Tenure  Usage Frequency  Support Calls  \
count    64374.00  64214.00  64308.00         64055.00       63927.00   
mean     32187.50     41.97     31.99            15.08           5.40   
std      18583.32     13.93     17.10             8.82           3.11   
min          1.00     18.00      1.00             1.00           0.00   
25%      16094.25     30.00     18.00             7.00           3.00   
50%      32187.50     42.00     33.00            15.00           6.00   
75%      48280.75     54.00     47.00            23.00           8.00   
max      64374.00     65.00     60.00            30.00          10.00   

       Payment Delay  Total Spend  Last Interaction     Churn  
count       63906.00     64325.00          64027.00  64027.00  
mean           17.15       541.01             15.50      0.48  
std             8.85       260.87              8.64      0.50  
min             0.00       100.00              1.00      0.00  
25%            10.00  

In [6]:
print(data.isnull().sum())

CustomerID             0
Age                  160
Gender               251
Tenure                66
Usage Frequency      319
Support Calls        447
Payment Delay        468
Subscription Type    278
Contract Length      144
Total Spend           49
Last Interaction     347
Churn                347
dtype: int64


# Fill Null Value Manually

In [7]:
# Fill Null Values of Gender
data.Gender = data.Gender.fillna(data.Gender.mode()[0])

# Fill Null Values of Age
data.Age = data.Age.fillna(data.Age.mean())

# Fill Null Values of Tenure
data.Tenure = data.Tenure.fillna(data.Tenure.mean())

# Fill Null Value of Usage Frequency
data["Usage Frequency"] = data["Usage Frequency"].fillna(data["Usage Frequency"].median())

# Fill Null Value of Support Calls
data["Support Calls"] = data["Support Calls"].fillna(data["Support Calls"].mean())

In [8]:
def smart_impute_missing_values(df):
    """
    Automatically detects and imputes missing values in a DataFrame.

    - **Categorical Columns (object, category):** Imputed with the **Mode**.
    - **Numerical Columns (int, float):** Imputed with the **Median** if
      outliers are detected (using IQR method) or the **Mean** otherwise.

    Args:
        df (pd.DataFrame): The input pandas DataFrame with missing values.

    Returns:
        pd.DataFrame: A new DataFrame with missing values imputed.
    """
    # Create a clean copy to work on, avoiding SettingWithCopyWarning
    df_imputed = df.copy()

    # 1. Identify columns with missing values
    missing_cols = df_imputed.columns[df_imputed.isnull().any()].tolist()
    print(f"Columns with missing values found: {missing_cols}\n")

    # 2. Iterate through columns with missing values
    for col in missing_cols:
        col_dtype = df_imputed[col].dtype
        null_count = df_imputed[col].isnull().sum()

        if pd.api.types.is_numeric_dtype(col_dtype):
            # --- NUMERICAL IMPUTATION (Mean/Median) ---

            # a. Outlier detection using IQR
            Q1 = df_imputed[col].quantile(0.25)
            Q3 = df_imputed[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            # Count of values outside the IQR range
            outlier_count = (
                (df_imputed[col] < lower_bound) |
                (df_imputed[col] > upper_bound)
            ).sum()

            if outlier_count > 0:
                # If outliers are present, use the MEDIAN (more robust)
                imputation_value = df_imputed[col].median()
                strategy = "Median (due to outliers)"
            else:
                # If few/no outliers, use the MEAN
                imputation_value = df_imputed[col].mean()
                strategy = "Mean (no significant outliers)"

            # Impute the missing values using assignment (Fixes the Warning)
            df_imputed[col] = df_imputed[col].fillna(imputation_value)

            print(f"✅ Imputed {null_count} nulls in **'{col}'** (Numerical) with: **{strategy}** -> Value: {imputation_value:.2f}")

        elif pd.api.types.is_object_dtype(col_dtype) or pd.api.types.is_categorical_dtype(col_dtype):
            # --- CATEGORICAL IMPUTATION (Mode) ---

            # Use the most frequent value (mode) for categorical data
            imputation_value = df_imputed[col].mode()[0]

            # Impute the missing values using assignment (Fixes the Warning)
            df_imputed[col] = df_imputed[col].fillna(imputation_value)

            print(f"✅ Imputed {null_count} nulls in **'{col}'** (Categorical) with: **Mode** -> Value: '{imputation_value}'")

        else:
            print(f"⚠️ Column '{col}' of dtype {col_dtype} skipped. Imputation not defined for this type.")

    print("\nImputation complete!")
    return df_imputed

# Fill Null Value Automatically

In [9]:
data = smart_impute_missing_values(data)

Columns with missing values found: ['Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']

✅ Imputed 468 nulls in **'Payment Delay'** (Numerical) with: **Mean (no significant outliers)** -> Value: 17.15
✅ Imputed 278 nulls in **'Subscription Type'** (Categorical) with: **Mode** -> Value: 'Standard'
✅ Imputed 144 nulls in **'Contract Length'** (Categorical) with: **Mode** -> Value: 'Monthly'
✅ Imputed 49 nulls in **'Total Spend'** (Numerical) with: **Mean (no significant outliers)** -> Value: 541.01
✅ Imputed 347 nulls in **'Last Interaction'** (Numerical) with: **Mean (no significant outliers)** -> Value: 15.50
✅ Imputed 347 nulls in **'Churn'** (Numerical) with: **Mean (no significant outliers)** -> Value: 0.48

Imputation complete!


In [8]:
print(data.isnull().sum())

CustomerID             0
Age                    0
Gender                 0
Tenure                 0
Usage Frequency        0
Support Calls          0
Payment Delay        468
Subscription Type    278
Contract Length      144
Total Spend           49
Last Interaction     347
Churn                347
dtype: int64


In [19]:
# Gemini Created
def missing_value(df):
    # Iterate only over columns that have missing values


In [22]:
def fill_missing(df):
    for col in df.columns[df.isnull().any()]:
        if pd.api.types.is_numeric_dtype(df[col]):
            q1, q3 = df[col].quantile([0.25, 0.75])
            IQR = q3 - q1
            lower_limit = q1 - IQR * 1.5
            upper_limit = q3 + IQR * 1.5
            outlier_present = ((df[col] < lower_limit) | (df[col] > upper_limit)).any()
            if outlier_present:
                fill_val = df[col].median()
            else:
                fill_val = df[col].mean()
        else:
            fill_val = df[col].mode()[0]

        df[col] = df[col].fillna(fill_val)

In [20]:
# Local Created
def fill_missing_value(df):
    for col in df.columns[df.isnull().any()]:
        if pd.api.types.is_numeric_dtype(df[col]):
            q1, q3 = df[col].quantile([0.25, 0.75])
            IQR = q3 - q1
            lower_limit = q1 - IQR * 1.5
            upper_limit = q3 + IQR * 1.5
            outlier_present = ((df[col] > upper_limit) | (df[col] < lower_limit))
            if ((len(outlier_present)/len(df[col])) > 0.05):
                fill_val = df[col].median()
            else:
                fill_val = df[col].mean()
        else:
            fill_val = df[col].mode()[0]
        df[col] = df[col].fillna(fill_val)
            

In [21]:
# Gemini Created
def fill_missing_value(df):
    for col in df.columns[df.isnull().any()]:
        if pd.api.types.is_numeric_dtype(df[col]):
            # 1. Calculate IQR on non-null values
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            IQR = q3 - q1
            
            lower_limit = q1 - IQR * 1.5
            upper_limit = q3 + IQR * 1.5
            
            # 2. Count how many outliers actually exist
            outlier_mask = (df[col] > upper_limit) | (df[col] < lower_limit)
            outlier_count = outlier_mask.sum() # Sums the True values
            
            # 3. Calculate percentage based on non-null values
            total_non_null = df[col].count()
            outlier_percentage = outlier_count / total_non_null
            
            # 4. Decide fill value
            if outlier_percentage > 0.05: # Using your 5% threshold
                fill_val = df[col].median()
            else:
                fill_val = df[col].mean()
        else:
            # Categorical: Use the first value of the mode
            mode_result = df[col].mode()
            fill_val = mode_result[0] if not mode_result.empty else "Unknown"
            
        df[col] = df[col].fillna(fill_val)
    return df

In [ ]:
def fill_missing(df):
    for col in df.columns[df.isnull().any]:
        if pd.api.types.is_numeric.dtype(df[col]):
            q1, q3 = df.quantile([0.25, 0.75])
            IQR = q3 - q1
            lower_limit = q1 - IQR * 1.5
            upper_limit = q3 + IQR * 1.5
            outlier_present = ((df[col] < lower_limit) | (df[col] > upper_limit))
            if outlier_present:
                fill_val = df[col].median()
            else:
                fill_val = df[col].mean()
        else:
            fill_val = df[col]

In [ ]:
# # Spacing Diffrence
# for i in range(1, 21):
#     print()
#     for j in range(1, 11):
#         print(f"{i:>2} * {j:>2} = {i*j:>2}")

# # 8 Tables on one Line
# for start in range(1, 21, 8):          # 1–5, 6–10, 11–15, 16–20
#     for j in range(1, 11):             # 1 to 10 multiples
#         for i in range(start, start + 8):
#             print(f"{i:>2} × {j:>2} = {i*j:>3}", end="   ")
#         print()
#     print()